```mermaid
graph
    %% Raw Dataset Source
    A[PBMC CITE-seq Data / Hao et al.]

    %% Modality Split

    subgraph modalities [" "]
        A --> RNA_MODALITY[RNA Modality]
        A --> ADT_MODALITY[ADT Modality]
    end

    %% Exploaratory analysis
    subgraph exploaratory ["Exploaratory analysis"]
        RNA_MODALITY --> PCA["Main variance drivers (PCA)"]
        RNA_MODALITY --> UMAP["Visualization in low dimention (UMAP)"]
        RNA_MODALITY --> BATCH_EFFECTS["Controlling for batch effects"]
    end


    %% Pre-processing and Normalization
    subgraph preprocessing [" "]
        PCA --> D1[log1p Normalization]
        UMAP --> D1[log1p Normalization]
        BATCH_EFFECTS --> D1[log1p Normalization]
        ADT_MODALITY --> D2[CLR Normalization]
    end

    %% Splitting
    E[MultiModal Dataset]
    D1 --> E
    D2 --> E
    E --> F[Data Split 40/60]

    %% 5. Split Blocks
    F -- Driver recovery methods --> G1[MultiModal test data]
    F -- MLP Training and tuning--> G2[MultiModal training data]

    %% Deep learning
    subgraph deep_learning [" "]
        G2 --> TUNING[Tuning]
        G2 --> TRAINING[Training]
    end

    %% Tuning
    TUNING --> TUNING_SUBSAMPLE[Subsample 10.000]
    TUNING_SUBSAMPLE --> TUNING_SPLIT[Split 15/85]
    TUNING_SPLIT --> TUNE[Tune using optuna\n n_trials = 50]
    TUNE --> PERSIST_PARAMS[Persist to file]

    %% Training
    TRAINING --> FETCH_PARAMS[Fetch tuned hyperparameters]
    FETCH_PARAMS --> TRAIN[Run training loop\n n_epochs = 15]
    TRAIN --> PERSIST_WEIGHTS[Persist weights to file]


    %% 6. Branching to the 4 Discovery Methods (Run strictly on Training Blocks)
    subgraph methods [" "]
        G1 -- Linear/Marginal --> M1["Method 1\n Spearman correlation"]
        G1 -- Linear/Conditional --> M2["Method 2\n Partial Corr \n (Leodit-Wolf covariance)"]
        G1 -- Non-Linear/Marginal --> M3["Method 3\n Mutual Information \n (KSG)"]
        G1 -- Non-Linear/Conditional --> M4["Method 4\n Deep Learning \n (MLP + IG)"]
    end


    subgraph transformation [" "]
        M1 --> SPEARMAN_TRANSFORMATION["No transformations \n (Computes ranks internally)"]
        M2 --> PARTIAL_CORR_TRANSFORMATION[Scale]
        M3 --> MI_TRANSFORMATION[No transformations]
        M4 --> MLP_TRANSFORMATION[Scale]
    end

    subgraph scoring [" "]
        SPEARMAN_TRANSFORMATION --> SPEARMAN_SCORING["Compute spearman"]
        PARTIAL_CORR_TRANSFORMATION --> PARTIAL_CORR_SCORING[Compute Ledoit-Wolf covariance matrix]
        MI_TRANSFORMATION --> MI_SCORING["Compute MI scores (sklearn)"]
        MLP_TRANSFORMATION --> MLP_SCORING[Compute Integrated Gradient Attributions]
    end

    GROUND_TRUTH["Established ground truth based on ADT modality and uniprot database"] --> DRIVER_DISCOVERY


    subgraph evaluation ["Evaluation"]
        SPEARMAN_SCORING --> DRIVER_DISCOVERY[Driver Gene Discovery Metric]
        PARTIAL_CORR_SCORING --> DRIVER_DISCOVERY
        MI_SCORING --> DRIVER_DISCOVERY
        MLP_SCORING --> DRIVER_DISCOVERY
    end

    subgraph annotation [" "]
        DRIVER_DISCOVERY --> DECOMPOSITION["Per-Gene decomposition analysis"]
        DRIVER_DISCOVERY --> GO_ENRICHMENT["GO-Enrichment analysis"]
    end
```